# Week10｜課程檢視、回顧與統整

Week06–09 複習，搭配 10 分鐘投影片。Colab：檔案 → 上傳筆記本；或用 VS Code Jupyter 開啟，依序執行。

需要 pandas、scikit-learn（Colab 通常已提供）。內建模擬資料，不需下載資料檔，不可用於真實估價。

## 暖身｜資料型態

list 儲存多筆資料，dict 用鍵值描述一筆資料。想一想：字串 "900" 和數字 900 是相同型態嗎？

In [2]:
prices = [900, 1200, 1500]
house = {"area": 20, "age": 10, "price": 900}


## 操作①｜CSV、JSON 與 Pandas（90 秒）

area＝坪數；age＝屋齡（年）；price＝房價（萬元）。

核對：8 列、3 欄、平均 1530 萬元，三欄均為數值。json.loads 讀字串，json.load 讀檔案。

In [3]:
import json
from io import StringIO
import pandas as pd

csv_text = """area,age,price
15,5,910
20,20,990
25,10,1270
30,30,1310
35,5,1710
40,15,1830
45,25,1950
50,10,2270
"""
df = pd.read_csv(StringIO(csv_text))
features = json.loads('{"columns": ["area", "age"]}')["columns"]
print(df.shape, df["price"].mean())
print(df.dtypes)


(8, 3) 1530.0
area     int64
age      int64
price    int64
dtype: object


In [4]:
# 將CSV存檔
df.to_csv("house.csv", index=False)


## 操作②｜訓練、預測與評估（120 秒）

X＝坪數、屋齡；y＝房價。6 筆訓練、2 筆測試。函式將流程打包，方便重複使用。

核對：28 坪、10 年屋齡 → 約 1390 萬元。

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

train, test = train_test_split(df, test_size=0.25, random_state=42)

def evaluate(columns):
    model = LinearRegression()
    model.fit(train[columns], train["price"])
    pred = model.predict(test[columns])
    print("特徵：", columns)
    print(f"MSE={mean_squared_error(test['price'], pred):.1f}")
    print(f"R²={r2_score(test['price'], pred):.3f}")
    return model

model = evaluate(features)
new_house = pd.DataFrame({"area": [28], "age": [10]})
print(f"預測：{model.predict(new_house)[0]:.0f} 萬元")


特徵： ['area', 'age']
MSE=0.0
R²=1.000
預測：1390 萬元


## 換你操作（90 秒）

先猜測，再執行下一格。

1. 少了屋齡，MSE 變大還是變小？
2. 為什麼只用坪數會漏掉資訊？
3. 程式如何確保兩次使用相同測試資料？

只改特徵，不重新切分。

In [10]:
area_only_model = evaluate(["area"])


特徵： ['area']
MSE=2550.4
R²=0.986


## 核對與限制

| 特徵 | MSE（萬元²） | R² |
| --- | --- | --- |
| 坪數＋屋齡 | 約 0 | 約 1.000 |
| 只有坪數 | 約 2550.4 | 約 0.986 |

- 兩次沿用同一組 train、test，沒有重新抽樣。
- MSE 越小越好；同一測試集的 R² 越大通常越好。R² 不是準確率，也可能小於 0。
- 資料刻意符合 price = 40 × area − 8 × age + 350，不能拿來真實估價。
- 只有 2 筆測試資料，不能推論真實效果；加特徵不一定更好。
- 真實專題應增加資料，以交叉驗證選模型，最後才評估保留的測試集。